<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex02-pytorch-and-autograd/Ex02_02_autograd_by_hand_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference text — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_2 · Notebook 02 — autograd, by hand

**This is the most important notebook in Part 1.** Everything you will do from
week seven to the end of the course is the two ideas below, applied to a
different equation each week.

## The claim

You already know that PyTorch computes gradients of a loss with respect to a
network's weights, and that an optimiser uses them to take a step. That is how
autograd is normally introduced, and it leaves almost everyone with the
impression that autograd is *a neural-network mechanism*.

It is not. It is a general differentiation engine. It will differentiate any
expression built from differentiable operations with respect to any tensor that
took part in it — and in particular with respect to the **input**.

So if `u` is the output of a network at coordinate `x`, then

```python
u_x  = grad(u, x)          # du/dx
u_xx = grad(u_x, x)        # d2u/dx2
```

are the first and second derivatives of the field the network represents. Exact
to machine precision. No finite differences, no step size, no mesh. Available
at any point you care to evaluate, including points where you have no data.

Now write down

```python
residual = u_xx + k**2 * u
```

and you have the Helmholtz equation, evaluated wherever you like. Minimise the
mean square of that residual with respect to the network's weights, and you
have solved a differential equation using **no data at all**. The loss contains
no measurements; it contains physics.

That is a physics-informed neural network, and it is the whole content of L7 to
L12. Ex_7.1 solves Poisson's equation this way. Ex_8 does heat conduction,
Ex_9 fluid flow, Ex_10 an electrochemical cell, Ex_12 power flow. Every one of
them is the two lines above with a different expression on the third line.

## What this notebook does

1. Differentiates a scalar function and checks it against the derivative you
   would write by hand.
2. Explains `grad_outputs`, which is the only piece of the interface that is
   not obvious.
3. Does the same for an analytic function on a grid, and checks every point.
4. Takes **second** derivatives, and shows what `create_graph=True` is for by
   removing it and watching the failure.
5. Compares autograd against finite differences over eleven decades of step
   size, which is the honest argument for why anyone bothers.
6. Builds a **residual** — the first physics-informed object in the course.
7. Differentiates a *network* with respect to its input.
8. Shows that the second derivative of a ReLU network is identically zero,
   while a tanh network's is not. This is why every network in Part 2 uses
   tanh, and it is the single most common cause of a physics-informed network
   that trains to a flat line and never improves.

Nothing here trains anything. There is no optimiser in this notebook. That is
deliberate: the mechanism has to be clear before the training loop obscures it.

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_2_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex02-pytorch-and-autograd/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup -------------------------------------------------------------
# Needs Ex_2_core.py alongside this notebook.
import os
for f in ("Ex_2_core.py",):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from Ex_2_core import *                             # noqa: F401,F403
import numpy as np
import torch
import matplotlib.pyplot as plt

set_seed(88)
torch.set_printoptions(precision=6, sci_mode=False)
print("setup complete | device:", DEVICE)

## 1 · One function, one point

Start with the smallest possible case.

Take f(x) = x³ − 2x. Its derivative is f'(x) = 3x² − 2, which you can write
down without thinking. At x = 2 that is 12 − 2 = 10.

Three lines produce it:

```python
x = torch.tensor([2.0], requires_grad=True)
y = x ** 3 - 2 * x
y_x = torch.autograd.grad(y, x)[0]
```

Read each one.

**`requires_grad=True`** marks `x` as a tensor autograd should track. Without
it, PyTorch records nothing about the operations `x` takes part in, and the
third line raises. This flag is the entire difference between an array library
and a differentiation engine, and it costs one keyword.

**`y = x ** 3 - 2 * x`** builds the graph as a side effect of computing the
value. Each operation adds a node that knows how to differentiate itself.

**`torch.autograd.grad(y, x)`** walks that graph backwards and returns a tuple
— hence the `[0]` — of gradients, one per input tensor requested.

What comes back is not a numerical approximation and not a symbolic expression.
It is the exact derivative of the operations that were actually executed,
obtained by applying the chain rule to them. If you had written a `for` loop
with an `if` in it, autograd would differentiate the branch that ran.

In [ ]:
x = torch.tensor([2.0], requires_grad=True)
y = x ** 3 - 2 * x
y_x = torch.autograd.grad(y, x)[0]

print("  x            ", x.item())
print("  y = x^3 - 2x ", y.item())
print("  autograd     ", y_x.item())
print("  3x^2 - 2     ", 3 * 2.0 ** 2 - 2)
print()
print("  difference   ", abs(y_x.item() - (3 * 2.0 ** 2 - 2)))

**What you should see.** `y = 4.0`, the autograd derivative `10.0`, the hand
derivative `10.0`, and a difference of exactly `0.0`.

Exactly zero, not nearly zero. Both numbers came from the same handful of
float32 multiplications, so there is nothing left to round.

![Computational graph of y = (x² + 2x)·sin x, with the backward pass](https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex02-pytorch-and-autograd/figures/computational_graph.png)

What the graph looks like for a richer expression, y = (x² + 2x)·sin x. Each circle is one operation that ran; the arrows are the forward pass. `x` fans out into three branches, so on the way back (dashed) three contributions arrive at `x` and are **added** — that addition is the bookkeeping autograd does for you. Check it: at x = 2, y = 7.274 and dy/dx = 2.127.

```python
x = torch.tensor([2.0], requires_grad=True)
y = (x**2 + 2*x) * torch.sin(x)
torch.autograd.grad(y, x)[0]      # tensor([2.1266])
```

## 2 · `grad_outputs`, and why a vector needs one

Now do the same thing at many points at once, and meet the one piece of the
interface that is not self-explanatory.

`x` is a column of n coordinates and `y` is a column of n values. Ask for
`torch.autograd.grad(y, x)` and PyTorch refuses: *grad can be implicitly created
only for scalar outputs*. The refusal is correct, and understanding why is worth
five minutes.

What autograd computes is not a derivative but a **vector–Jacobian product**.
Given outputs y and inputs x, the Jacobian J has entry J[i, j] = ∂y_i/∂x_j.
Autograd never forms J — for a real problem it would be enormous — and instead
computes vᵀJ for a vector v that you supply. That single operation is what a
backward pass is, and it is what makes backpropagation cheap.

When y is a scalar, there is only one sensible v, namely 1, and PyTorch fills it
in for you. When y is a vector, it cannot guess, so you say:

```python
torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y))
```

**Why ones is the right choice here.** Our map is elementwise: y_i depends only
on x_i. So J is diagonal, with J[i, i] = dy_i/dx_i and zeros elsewhere.
Multiplying by a vector of ones sums each column, and a column of a diagonal
matrix contains exactly one non-zero entry. The result is the vector of
pointwise derivatives — one derivative per sample, which is what a residual
needs at each collocation point.

If the map were not elementwise, a vector of ones would give you the sum of the
rows of the Jacobian, which is a different and usually meaningless quantity.
Every coordinate tensor in this course goes through a network one sample at a
time, so the map is elementwise and ones is correct. It is worth knowing why
rather than copying it.

The core module wraps all of this:

```python
def grad(y, x):
    return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y),
                               create_graph=True, retain_graph=True,
                               allow_unused=True)[0]
```

That function, under that name, appears in the core library of every Part 2
exercise. Use it from here on.

### Your turn: the same derivative, on a grid

Compute the derivative of f(x) = x³ − 2x at eleven points from −2 to 2, and
compare with the hand derivative.

You need:

| name | requirement |
|---|---|
| `xg` | eleven points from −2 to 2, shape `(11, 1)`, requiring grad |
| `yg` | f applied to `xg`, shape `(11, 1)` |
| `yg_x` | the derivative from autograd, shape `(11, 1)` |
| `yg_x_hand` | 3x² − 2 evaluated at `xg`, shape `(11, 1)` |

`as_input(...)` from the core module builds the coordinate for you: it makes a
column of shape `(n, 1)`, in the course dtype, on the course device, with
`requires_grad=True`. Use `as_input(torch.linspace(-2, 2, 11))`.

For `yg_x`, use the core `grad(yg, xg)`. For `yg_x_hand`, write the formula in
torch operations on `xg` — this is a comparison of two computations of the same
quantity, so both live in tensor land.

A subtlety worth noticing: `yg_x_hand` will carry a `grad_fn` too, because it
was computed from a tensor that requires grad. That is harmless. `check`
detaches before comparing.

In [ ]:
# TODO 1 --- the same derivative, on a grid ------------------------------------------
# Four `...` to replace, one per line:
#   xg         ->  as_input(torch.linspace(-2.0, 2.0, 11))    column, requires grad
#   yg         ->  xg ** 3 - 2.0 * xg                          f(x) = x^3 - 2x
#   yg_x       ->  grad(yg, xg)                                autograd
#   yg_x_hand  ->  3.0 * xg ** 2 - 2.0                         by hand
xg        = ...                                   # <- as_input(torch.linspace(-2.0, 2.0, 11))
yg        = ...                                   # <- xg ** 3 - 2.0 * xg
yg_x      = ...                                   # <- grad(yg, xg)
yg_x_hand = ...                                   # <- 3.0 * xg ** 2 - 2.0
assert not any(v is ... for v in (xg, yg, yg_x, yg_x_hand)), "TODO 1: replace the four ..."
# ------------------------------------------------------------------------------

In [ ]:
check_shape("xg", xg, (11, 1))
check_shape("yg_x", yg_x, (11, 1))
check("autograd matches the hand derivative", yg_x, yg_x_hand, tol=1e-5)

print()
print("      x        f(x)      autograd    by hand")
for i in range(0, 11, 2):
    print(f"  {xg[i, 0].item():+6.2f}  {yg[i, 0].item():+9.4f}  "
          f"{yg_x[i, 0].item():+9.4f}  {yg_x_hand[i, 0].item():+9.4f}")

**What you should see.** Three `PASS` lines and a table of six rows in which
the last two columns are identical. At x = −2 the derivative is 10, at x = 0 it
is −2, at x = 2 it is 10.

If you got `RuntimeError: grad can be implicitly created only for scalar
outputs`, you called `torch.autograd.grad` directly without `grad_outputs`
rather than using the core `grad`. That error message is worth recognising; it
means "tell me which vector–Jacobian product you want".

## 3 · An analytic function, and a derivative worth checking

A cubic is too forgiving. Take something with an exponential and a
trigonometric factor, where the product rule can go wrong:

$$u(x) = e^{-x}\,\sin(3x)$$

Differentiate it by hand, now, before running anything. The product rule gives

$$u'(x) = e^{-x}\bigl(3\cos(3x) - \sin(3x)\bigr)$$

and differentiating again, collecting terms,

$$u''(x) = e^{-x}\bigl(-8\sin(3x) - 6\cos(3x)\bigr)$$

Check that second one for yourself: differentiate u′ with the product rule,
you get e^{-x}(−9 sin 3x − 3 cos 3x) − e^{-x}(3 cos 3x − sin 3x), and
collecting the sine and cosine terms gives −8 sin 3x − 6 cos 3x. The point of
doing this by hand is that in the next section you will compare autograd
against it, and a comparison is only worth something if you trust both sides.

**Write the function in `torch` operations, not NumPy ones.** `torch.exp` and
`torch.sin` are differentiable and record themselves on the graph. `np.exp` of
a tensor either raises or silently escapes into NumPy, where autograd cannot
follow. This is the single most common way a first PINN implementation fails
silently.

In [ ]:
def u_fn(x):
    """u(x) = exp(-x) sin(3x), in torch operations so autograd can follow."""
    return torch.exp(-x) * torch.sin(3.0 * x)


def u_x_hand(x):
    """The hand-derived first derivative."""
    return torch.exp(-x) * (3.0 * torch.cos(3.0 * x) - torch.sin(3.0 * x))


def u_xx_hand(x):
    """The hand-derived second derivative."""
    return torch.exp(-x) * (-8.0 * torch.sin(3.0 * x) - 6.0 * torch.cos(3.0 * x))


x = as_input(torch.linspace(0.0, 2.0, 101))
print("x:", tuple(x.shape), x.dtype, "requires_grad =", x.requires_grad)

In [ ]:
# TODO 2 --- first derivative of exp(-x) sin(3x) ---------------------------------------
# Three `...` to replace, one per line:
#   u        ->  u_fn(x)          the values on the grid
#   u_x      ->  grad(u, x)       autograd
#   u_x_ref  ->  u_x_hand(x)      the hand derivative
u       = ...                                     # <- u_fn(x)
u_x     = ...                                     # <- grad(u, x)
u_x_ref = ...                                     # <- u_x_hand(x)
assert not any(v is ... for v in (u, u_x, u_x_ref)), "TODO 2: replace the three ..."
# ------------------------------------------------------------------------------

In [ ]:
check_shape("u", u, (101, 1))
check_shape("u_x", u_x, (101, 1))
check("first derivative, autograd vs hand", u_x, u_x_ref, tol=1e-4)
print(f"  largest |u'| on the grid: {u_x.abs().max().item():.4f}")

**What you should see.** Three `PASS` lines, with a max absolute error around
1e-6 and a largest derivative of about 3.

**Why the tolerance is 1e-4 and not 1e-12.** These are `float32` tensors, which
carry about seven significant decimal digits. Two different routes to the same
quantity — the chain rule applied operation by operation, and the closed-form
expression — round differently at the seventh digit, and on values of order 3
that shows up around 1e-6. Asking for agreement to 1e-12 would be asking the
arithmetic for precision it does not have.

That is a general habit worth forming: a tolerance should be set by the
precision of the arithmetic and the size of the quantity, not by optimism.

## 4 · Second derivatives, and what `create_graph` is for

A first derivative is useful. A second derivative is the point, because almost
every conservation law in engineering is second order: heat conduction,
diffusion, the wave equation, elasticity, Poisson's equation, viscous momentum
transport. If you cannot get u″ you cannot write any of them.

Getting it requires one extra idea. By default, PyTorch computes a gradient and
then throws the machinery away — the backward pass is not itself recorded, so
the result is a plain tensor with no history and differentiating it again is
impossible.

`create_graph=True` changes that. The operations performed *during* the backward
pass are recorded too, so the gradient is itself a differentiable function of
`x`. You can then call `grad` on it, and again, as many times as you like. The
core `grad` sets this flag, which is why `d2` is simply

```python
def d2(y, x):
    return grad(grad(y, x), x)
```

The cost is memory and time: the second graph is roughly as large as the first,
and each further derivative adds another. For the second-order problems in this
course that is entirely affordable. For a fourth-order plate equation it starts
to hurt, which is worth knowing before you meet one.

In [ ]:
# TODO 3 --- second derivative ------------------------------------------------------
# Two `...` to replace, one per line:
#   u_xx      ->  d2(u, x)          autograd, twice
#   u_xx_ref  ->  u_xx_hand(x)      the hand second derivative
u_xx     = ...                                    # <- d2(u, x)
u_xx_ref = ...                                    # <- u_xx_hand(x)
assert not any(v is ... for v in (u_xx, u_xx_ref)), "TODO 3: replace the two ..."
# ------------------------------------------------------------------------------

In [ ]:
check_shape("u_xx", u_xx, (101, 1))
check("second derivative, autograd vs hand", u_xx, u_xx_ref, tol=1e-3)
print(f"  largest |u''| on the grid: {u_xx.abs().max().item():.4f}")

fig, ax = plt.subplots(figsize=(6.5, 4.0))
xn = to_numpy(x).ravel()
ax.plot(xn, to_numpy(u).ravel(), "-", label="u")
ax.plot(xn, to_numpy(u_x).ravel(), "--", label="u' (autograd)")
ax.plot(xn, to_numpy(u_xx).ravel(), "-.", label="u'' (autograd)")
ax.plot(xn, to_numpy(u_xx_ref).ravel(), ":", linewidth=2.5, label="u'' (by hand)")
engineering_axes(ax, "x [-]", "value [-]",
                 title="a function and its first two derivatives", legend=True)
plt.show()

**What you should see.** Two `PASS` lines with an error of order 1e-5, a
largest second derivative near 9, and a figure in which the dotted hand-derived curve
lies exactly on top of the dash-dotted autograd one.

Note that the tolerance loosened again, from 1e-4 to 1e-3. Each differentiation
amplifies the rounding error of the one before it, and the values themselves
grew — u″ reaches 9 where u reaches 1. Both effects push the absolute error up.
Nothing is wrong; the arithmetic is doing what float32 arithmetic does.

### What happens without `create_graph`

Worth seeing once, because the error message is not obvious and you will
eventually cause it.

The cell below takes a first derivative with `create_graph=False` — the default
— and then tries to differentiate the result. Read the exception it produces.

In [ ]:
x2 = as_input(torch.linspace(0.0, 2.0, 5))
u2 = u_fn(x2)

u2_x = torch.autograd.grad(u2, x2, grad_outputs=torch.ones_like(u2),
                           create_graph=False)[0]          # the graph is discarded

print("  u2_x requires_grad:", u2_x.requires_grad, " grad_fn:", u2_x.grad_fn)
print()
try:
    torch.autograd.grad(u2_x, x2, grad_outputs=torch.ones_like(u2_x))
except RuntimeError as exc:
    print("  RuntimeError:", str(exc)[:150])

**What you should see.** `u2_x requires_grad: False`, `grad_fn: None`, and then
a `RuntimeError` saying that the tensor *does not require grad and does not have
a grad_fn*.

That message means precisely one thing: **you asked for the derivative of
something that is not connected to anything differentiable.** In a
physics-informed network it means you forgot `create_graph=True` on the first
call. It is the second most common failure in a first PINN, after using ReLU —
which is §8.

## 5 · Why not finite differences?

The honest question. You have known how to approximate a derivative since your
first numerical methods course:

$$u'(x) \approx \frac{u(x+h) - u(x-h)}{2h}$$

It is two function evaluations and one division. Why build a graph?

Three answers, in increasing order of importance.

**It has an h, and there is no good choice for it.** The central difference has
a truncation error that falls like h², so small h is good. It also subtracts two
nearly equal numbers and divides by a small one, so floating-point cancellation
grows like ε/h, and small h is bad. The total error therefore has a minimum at
some h that depends on the function, the point, and the precision, and either
side of it the answer degrades. The cell below measures this.

**It costs two evaluations per derivative per dimension.** For a scalar problem
that is nothing. For the second derivative of a three-dimensional field you need
a stencil in every direction and every mixed pair, and each evaluation is a
forward pass through a network. Autograd gets all of them from one backward
pass.

**It is not exact, and the loss you are minimising is made of it.** In a
physics-informed network the residual *is* the loss. An approximate derivative
means you are minimising an approximation to your physics, and the error floor
of your solution is set by the error floor of your derivative — regardless of
how well the optimiser does its job.

The experiment: sweep h over eleven decades at a single point, and compare with
autograd. This section works in **float64** on purpose, because the whole
question is about numerical precision and float32 would hide the interesting
part.

In [ ]:
def u_np(x):
    return np.exp(-x) * np.sin(3.0 * x)


def u_x_exact(x):
    return np.exp(-x) * (3.0 * np.cos(3.0 * x) - np.sin(3.0 * x))


x0 = 0.7
exact = float(u_x_exact(np.array(x0)))
hs = np.logspace(-1, -12, 34)

# autograd, in float64 so the comparison is about method and not about dtype
xt = torch.tensor([x0], dtype=torch.float64, requires_grad=True)
ut = torch.exp(-xt) * torch.sin(3.0 * xt)
autograd_value = torch.autograd.grad(ut, xt)[0].item()
autograd_error = max(abs(autograd_value - exact), 1e-17)   # floored so it can be plotted

print(f"  exact          {exact:.15f}")
print(f"  autograd       {autograd_value:.15f}")
print(f"  autograd error {autograd_error:.2e}")

In [ ]:
# TODO 4 --- the finite-difference error for every h -----------------------------------
# One `...` to replace, inside the list comprehension:
#   abs(finite_difference(u_np, x0, h) - exact)
fd_errors = [... for h in hs]                     # <- abs(finite_difference(u_np, x0, h) - exact)
assert fd_errors[0] is not ..., "TODO 4: replace the ..."
# ------------------------------------------------------------------------------

In [ ]:
fd_errors = np.asarray(fd_errors, dtype=float).ravel()
check_shape("fd_errors", fd_errors, (len(hs),))

best = int(np.argmin(fd_errors))
print(f"  best finite difference: h = {hs[best]:.1e}, error {fd_errors[best]:.2e}")
print(f"  autograd:                          error {autograd_error:.2e}")
print(f"  and autograd needed no choice of h at all")

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.loglog(hs, fd_errors, "o-", markersize=3, label="central difference")
ax.axhline(autograd_error, linestyle="--", color="k", label="autograd")
ax.set_xscale("log")
engineering_axes(ax, "step size h [-]", "absolute error in u'(0.7) [-]",
                 title="the finite-difference trade-off, in float64", legend=True)
ax.invert_xaxis()
plt.show()

**What you should see.** A V-shaped — or rather a check-mark-shaped — curve.
Coming from large h on the left, the error falls like h² as truncation error
shrinks. Somewhere near h = 1e-5 or 1e-6 it reaches a minimum of about 1e-11.
Below that it climbs again, roughly like 1/h, as cancellation takes over: at
h = 1e-12 the two function values agree in the first eleven digits and the
subtraction keeps almost nothing.

The autograd line sits flat near machine epsilon, across the whole plot,
because it never subtracts two nearly equal numbers and never had an h to
choose.

Read the two error floors carefully. The best finite difference, with h chosen
by hindsight, is about 1e-11. Autograd is about 1e-16. And in a real problem
you do not get the hindsight — you have to pick h in advance, for a function
whose scale you do not yet know, at every point of a domain where the right
choice differs.

## 6 · A residual: the first physics-informed object in the course

Now put it together, and meet the idea the second half of this course is built
on.

The Helmholtz equation in one dimension is

$$u''(x) + k^2 u(x) = 0$$

It describes a standing wave — an acoustic mode in a duct, a vibrating string,
a beam in free harmonic motion. Its solutions are sin(kx) and cos(kx).

Define the **residual** as the left-hand side:

$$r(x) = u''(x) + k^2 u(x)$$

For an exact solution the residual is zero everywhere. For anything else it is
not, and how far it is from zero measures how badly the function fails to
satisfy the physics. That single sentence is the whole conceptual jump of Part
2: **a residual is a loss, and it contains no data.**

Write `helmholtz_residual(u, x, k)`. It takes a value tensor `u` that was
computed from `x`, and returns `d2(u, x) + k**2 * u`. Two lines including the
`return`.

Then evaluate it twice:

- on `u = sin(kx)`, an exact solution — the residual should be zero to
  arithmetic precision;
- on `u = sin(1.7 k x)`, which is a perfectly nice smooth function and is
  simply not a solution — the residual should be large.

Nothing here is trained, and nothing here is fitted to data. You are evaluating
how well a candidate function satisfies a differential equation, at whatever
points you choose, using nothing but the function itself.

In [ ]:
K = 3.0
xr = as_input(torch.linspace(0.0, 1.0, 201))

# TODO 5 --- the Helmholtz residual --------------------------------------------------
# One `...` to replace, in the return line:  d2(u, x) + k ** 2 * u
def helmholtz_residual(u, x, k):
    """The left-hand side of u'' + k^2 u = 0, evaluated where u was computed."""
    return ...                                    # <- d2(u, x) + k ** 2 * u
# ------------------------------------------------------------------------------

In [ ]:
u_true = torch.sin(K * xr)
r_true = helmholtz_residual(u_true, xr, K)

u_wrong = torch.sin(1.7 * K * xr)
r_wrong = helmholtz_residual(u_wrong, xr, K)

check_shape("residual", r_true, (201, 1))
print(f"  max |r| for u = sin(kx)      : {r_true.abs().max().item():.3e}   <- a solution")
print(f"  max |r| for u = sin(1.7 k x) : {r_wrong.abs().max().item():.3e}   <- not a solution")
print(f"  mean square residual, solution     : {(r_true ** 2).mean().item():.3e}")
print(f"  mean square residual, non-solution : {(r_wrong ** 2).mean().item():.3e}")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(to_numpy(xr).ravel(), to_numpy(r_true).ravel(), "-", label="u = sin(kx)")
ax.plot(to_numpy(xr).ravel(), to_numpy(r_wrong).ravel(), "--", label="u = sin(1.7kx)")
engineering_axes(ax, "x [-]", "residual  u'' + k^2 u  [-]",
                 title="a residual is a loss, and it contains no data", legend=True)
plt.show()

**What you should see.** A residual of order 1e-5 to 1e-4 for the true
solution — zero, to float32 precision, on values whose second derivative
reaches 9 — and a residual of order 10 for the impostor. The figure shows one flat line on zero
and one large oscillation.

Now consider what you would do next if `u` came from a network with adjustable
weights instead of from `torch.sin`. You would minimise the mean square
residual with respect to those weights. Nothing else about the machinery would
change.

**That is Ex_7.1.** The residual is Poisson's rather than Helmholtz's, boundary
conditions are added as a second loss term, and the optimiser is Adam followed
by L-BFGS. The line that does the work is the one you just wrote.

## 7 · The derivative of a network with respect to its input

The last step is to replace `torch.sin` with a network and confirm that nothing
breaks.

A network is a function. It is built from matrix multiplications and
activations, all differentiable, so autograd handles it exactly as it handled
the exponential and the sine. `grad(net(x), x)` is the derivative of the field
the network represents.

The network below is **untrained**. Its weights are random, so the function it
represents is an arbitrary smooth wiggle that means nothing. That is fine and
it is worth dwelling on: the derivative of a meaningless function is still a
well-defined object, and autograd returns it exactly. Training is what makes
the function meaningful; differentiation does not wait for it.

Your task: compute `un`, `un_x` and `un_xx`, and then verify `un_x` against a
central difference of the network — because at this point in the notebook you
should trust neither of them without a check.

For the finite-difference comparison, evaluate the network inside
`torch.no_grad()`. You are not differentiating it there, and switching the
recorder off makes it cheaper and makes your intent clear.

In [ ]:
set_seed(88)
net = mlp(n_in=1, n_out=1, n_hidden=32, n_layers=3, activation="tanh")
print("  network:", count_parameters(net), "parameters, tanh activation")

xn = as_input(torch.linspace(-1.0, 1.0, 201))

In [ ]:
# TODO 6 --- a network and its first two derivatives -----------------------------------
# Three `...` to replace, one per line:
#   un     ->  net(xn)
#   un_x   ->  grad(un, xn)
#   un_xx  ->  d2(un, xn)
un    = ...                                       # <- net(xn)
un_x  = ...                                       # <- grad(un, xn)
un_xx = ...                                       # <- d2(un, xn)
assert not any(v is ... for v in (un, un_x, un_xx)), "TODO 6: replace the three ..."
# ------------------------------------------------------------------------------

In [ ]:
check_shape("un", un, (201, 1))
check_shape("un_x", un_x, (201, 1))
check_shape("un_xx", un_xx, (201, 1))

# an independent check: central difference of the network itself
h = 1e-2
with torch.no_grad():
    fd = (net(xn + h) - net(xn - h)) / (2 * h)
check("autograd vs central difference of the network", un_x, fd, tol=5e-3)

fig, ax = plt.subplots(figsize=(6.5, 4.0))
xx = to_numpy(xn).ravel()
ax.plot(xx, to_numpy(un).ravel(), "-", label="u (untrained network)")
ax.plot(xx, to_numpy(un_x).ravel(), "--", label="du/dx")
ax.plot(xx, to_numpy(un_xx).ravel(), "-.", label="d2u/dx2")
engineering_axes(ax, "x [-]", "value [-]",
                 title="an untrained tanh network and its derivatives", legend=True)
plt.show()

**What you should see.** Four `PASS` lines and three smooth curves. The network
output is a gentle wiggle of order 0.1; its derivatives are larger and wigglier,
as derivatives are.

The finite-difference check passes with a tolerance of 5e-3, which is far
looser than anything in this notebook so far. Both error sources from §5 are
present: truncation, of order h²·u‴/6 ≈ 1e-4·u‴, and float32 cancellation, of
order ε·|u|/h ≈ 1e-7/1e-2 = 1e-5. The autograd value is the accurate one and
the finite difference is the approximation being checked against it — which is
the opposite of how the comparison usually feels the first time you write it.

## 8 · ReLU has no second derivative, and that decides the whole course

Here is the punchline, and the reason L4.1 spends a slide on activation
functions eight weeks before you need one.

A network with ReLU activations is **piecewise linear**. This is not an
approximation or a figure of speech: composing linear maps with
max(0, ·) produces a function made of flat pieces joined at kinks, exactly as
the region-counting picture in L4.1 shows. Its first derivative is piecewise
constant. Its second derivative is therefore **zero on every piece**, and
undefined at the kinks — where mathematically it is a Dirac delta, and where
autograd, which does arithmetic rather than distribution theory, simply
reports zero.

Consequence: put a ReLU network into a second-order residual and the u″ term is
identically zero, everywhere, for every setting of the weights. The gradient of
the loss with respect to the weights through that term is zero as well. The
optimiser gets no signal from the physics. The loss flattens out at a value
that has nothing to do with the equation, and the network converges to
something wrong with no error message anywhere.

A tanh network is smooth. Every derivative exists, is non-zero, and depends on
the weights — so the optimiser can act on it.

**This is why every network in L7 to L12 uses tanh.** One slide in L4.1, one
line in every core library, and it prevents the single most common
implementation failure in physics-informed learning.

Your task: write `derivatives(net, x)`, returning the triple `(u, u_x, u_xx)`.
Then the cell below applies it to two networks that are identical apart from
their activation, and prints the largest second derivative each one produces.

![ReLU and tanh with their first and second derivatives](https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex02-pytorch-and-autograd/figures/activations_derivatives.png)

The three rows are the function, its first derivative and its second derivative; ReLU on the left, tanh on the right. The bottom row is the one that decides: ReLU's second derivative is exactly zero everywhere it exists, tanh's is of order one. Your `derivatives` function below should reproduce exactly this contrast, point by point.

In [ ]:
# TODO 7 --- the same three lines as a function --------------------------------------
# Three `...` to replace, one per line:
#   u     ->  net(x)
#   u_x   ->  grad(u, x)
#   u_xx  ->  d2(u, x)
def derivatives(net, x):
    """Return (u, u_x, u_xx): the network's output and its first two derivatives."""
    u    = ...                                    # <- net(x)
    u_x  = ...                                    # <- grad(u, x)
    u_xx = ...                                    # <- d2(u, x)
    return u, u_x, u_xx
# ------------------------------------------------------------------------------

In [ ]:
set_seed(88)
net_tanh = mlp(n_in=1, n_out=1, n_hidden=32, n_layers=3, activation="tanh")
set_seed(88)
net_relu = mlp(n_in=1, n_out=1, n_hidden=32, n_layers=3, activation="relu")

xa = as_input(torch.linspace(-1.0, 1.0, 401))
u_t, ux_t, uxx_t = derivatives(net_tanh, xa)
u_r, ux_r, uxx_r = derivatives(net_relu, xa)

banner("largest |d2u/dx2| over the grid")
print(f"  tanh network : {uxx_t.abs().max().item():.6e}")
print(f"  relu network : {uxx_r.abs().max().item():.6e}")
print()
print(f"  relu second derivative exactly zero everywhere: "
      f"{bool((uxx_r == 0).all())}")

fig, axes = plt.subplots(2, 3, figsize=(12.0, 6.0), sharex=True)
xx = to_numpy(xa).ravel()
for row, (name, trio) in enumerate([("tanh", (u_t, ux_t, uxx_t)),
                                    ("relu", (u_r, ux_r, uxx_r))]):
    for col, (lbl, val) in enumerate(zip(["u", "du/dx", "d2u/dx2"], trio)):
        axes[row, col].plot(xx, to_numpy(val).ravel())
        engineering_axes(axes[row, col], "x [-]", f"{lbl} [-]",
                         title=f"{name}: {lbl}")
fig.tight_layout()
plt.show()

**What you should see.** A tanh second derivative of order 1, and a ReLU second
derivative of **exactly** `0.000000e+00`, with `True` on the line that tests it.

The figure makes the reason visible. The top row is three smooth curves. The
bottom row is a piecewise-linear function, then a staircase of constant pieces,
then a flat line on zero. Look at the middle panel of the bottom row: the
derivative jumps between levels and is constant in between. The derivative of a
constant is zero, and that is the whole argument.

### A detail about how autograd reports it

For the ReLU network, the first derivative does not depend on `x` anywhere in
the graph — it is a product of weight matrices with a constant mask, and the
mask came from a comparison, which is not differentiable. So a second `grad`
call finds no path back to `x` at all. Left alone, PyTorch raises *One of the
differentiated Tensors appears to not have been used in the graph*.

The core `grad` passes `allow_unused=True` and substitutes zeros, which is the
mathematically right answer and keeps this cell running. The price of that
convenience is worth stating: a derivative that comes back exactly zero might
mean "the derivative is zero" or might mean "you differentiated with respect to
the wrong tensor". If you ever see an exact zero you did not expect, check that
your output really does depend on the input you differentiated with respect to.

### Say it once more, because it is examinable and it is practical

A second-order PDE residual asks a network for curvature. ReLU networks have
none. Use `tanh`.

## 9 · The failure modes, collected

Every autograd problem you will have this semester is on this list.

| symptom | cause | fix |
|---|---|---|
| `element 0 of tensors does not require grad` | the input was not marked | `requires_grad=True`, or use `as_input` |
| `grad can be implicitly created only for scalar outputs` | vector output, no `grad_outputs` | pass `torch.ones_like(y)`, or use the core `grad` |
| `Trying to backward through the graph a second time` | the graph was freed | `retain_graph=True`, or restructure so one forward pass is used once |
| second derivative raises, or is `None` | `create_graph=False` on the first call | use the core `grad`, which sets it |
| gradient is exactly zero | ReLU, or the output does not depend on that input | use `tanh`; check the dependency |
| gradient is `None` | `.detach()`, `.item()`, `.numpy()` or `no_grad()` in the middle | keep the whole computation in tensors |
| the loss is a tensor of many values | you forgot to reduce it | `.mean()` — `backward()` needs a scalar |
| a NumPy function in the middle of the residual | `np.sin` instead of `torch.sin` | use `torch.` functions throughout |

## 10 · Where this goes

Below is the shape of every exercise from Ex_7 onwards, in eight lines. You
have now written all of the middle of it.

```python
u = model(x)                          # the trial solution
u_x = grad(u, x)                      # first derivative w.r.t. the INPUT
u_xx = grad(u_x, x)                   # second derivative

residual = u_xx - f(x)                # the physics. Poisson, here
loss = (residual ** 2).mean()         # a loss with no data in it

loss.backward()                       # gradients w.r.t. the WEIGHTS
optimizer.step()                      # ... and a step
```

Three of those lines are the subject of this notebook. The last two are
notebook 03. The one that changes from week to week is `residual`, and it is
the only one that carries any engineering.

## What you have done

You have differentiated an analytic function with respect to its input and
checked the result against a derivative you produced by hand; you know what
`grad_outputs` selects and why ones is the right choice for an elementwise map;
you have taken second derivatives and seen exactly what `create_graph=True`
buys; you have measured autograd against finite differences over eleven decades
of step size; you have built a residual and confirmed it is zero for a solution
and large for a non-solution; you have differentiated a network with respect to
its input; and you have seen a ReLU network return a second derivative of
exactly zero and can say why.

That is the mechanism of the entire second half of this course.

## Next

`Ex02_03_training_loop.ipynb` — forward, loss, `zero_grad`, `backward`, `step`.
The other half of the eight lines above.